In [309]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
from engression.loss_func import energy_loss, energy_loss_two_sample

In [179]:
df = pd.read_csv('PoliticalDemocracy.csv')
var_names = df.columns.values.tolist()
dataset = torch.tensor(df.to_numpy()).to(dtype = torch.float32)
batch_size = 25
dataloader = DataLoader(TensorDataset(dataset), batch_size = batch_size, shuffle = True)
print(var_names)

['y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'x1', 'x2', 'x3']


In [3]:
observation = next(iter(dataloader))
print(observation)

[tensor([[ 9.2000,  0.0000,  9.9000,  3.3333,  7.6097,  0.0000,  8.1188,  3.3333,
          4.3820,  2.8904,  1.7113],
        [ 2.5000,  0.0000,  6.6667,  3.3333,  2.5000,  0.0000,  0.0000,  0.0000,
          3.8067,  2.0794,  2.1376],
        [ 2.5000,  0.0000,  6.6667,  4.4333,  5.0000,  0.0000,  6.6667,  1.4852,
          5.0752,  5.2523,  5.3507],
        [ 3.3000,  0.0000,  6.6667,  3.3333,  6.2500,  0.0000,  6.6667,  3.3333,
          5.1417,  4.5643,  2.2551],
        [ 7.8000, 10.0000,  6.6667,  6.6667,  5.0000,  3.3333,  6.6667,  6.6667,
          5.0499,  5.1120,  4.3815],
        [ 5.0000,  3.3333,  3.3333,  2.2333,  5.0000,  3.3333,  6.6667,  5.5667,
          5.2575,  5.8406,  5.0018],
        [ 1.2500,  0.0000,  4.7000,  0.7370,  2.5000,  0.0000,  3.3333,  3.3333,
          4.8598,  5.6699,  3.5374],
        [ 7.5000,  3.3333,  6.6667,  6.6667,  8.7500,  3.3333,  6.6667,  6.6667,
          5.3982,  6.2461,  4.5357],
        [ 2.5000,  0.0000,  3.3333,  3.3333,  2.5000,  

In [4]:
print(var_names)

['y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'x1', 'x2', 'x3']


In [115]:
import scipy.stats as stats
from graphlib import TopologicalSorter, _NodeInfo

class rv_sem2(_NodeInfo):
    def __init__(self, formula: str):

        if "<-" in formula:
            lhs, rhs = formula.replace(" ", "").split("<-")
            super().__init__(lhs) # lhs is the variable name
            self.predecessors = rhs.split("+") # rhs is the parents of the node
            self.dist = None
            self.is_random = False
        elif "~" in formula:
            lhs, rhs = formula.replace(" ", "").split("~")
            super().__init__(lhs) # lhs is the variable name
            self.predecessors = []
            self.is_random = True
            self.dist = eval("stats."+rhs+"()") # rhs is the distribution (must be a class in scipy.stats)
        else:
            raise ValueError("Formula does not contain <- or ~.")

        self.npredecessors = len(self.predecessors) # NOTE: this will be changed when obtaining topological order
        self.formula = formula

    def __repr__(self):
        return self.formula

In [146]:
import torch
from torch import nn
class sem2(TopologicalSorter, nn.Module):
    def __init__(self, model_formula = None):

        nn.Module.__init__(self)
        TopologicalSorter.__init__(self, graph = None)

        formula_list = model_formula.replace(" ", "").split(";")

        # Initialize node info by formula
        var = [rv_sem2(formula) for formula in formula_list]
        self._node2info = {v.node:v for v in var}                
        
        for node, v in self._node2info.items():                
            for p in v.predecessors:
                self._node2info[p].successors.append(node)

        self.layer = nn.ModuleDict()
        for node, v in self._node2info.items():
            if v.is_random is True:
                self.layer[node] = None
            else:
                self.layer[node] = nn.Linear(len(v.predecessors), 1)

        self._topological_order = list(super(sem2, self).static_order())
        self._node2index = {node: index for index, node in enumerate(self._topological_order)}
    
    def _get_nodeinfo(self, node: str):
        """Override original private method, raise an error if node does not exist
        """
        return self._node2info[node]

    def add(self, node, *predecessors):
        """Override original methods, don't do anything
        """
        return None

    def static_order(self):
        """Override static_order to return cached topological order"""
        return iter(self._topological_order)

    def forward(self, batch_size = 1):
        n_nodes = len(self._node2info)
        x = torch.zeros(batch_size, n_nodes)
        for node in self._topological_order:
            v = self._node2info[node]
            index = self._node2index[node]
            if v.is_random is True:
                 x[:, index] = torch.tensor(v.dist.rvs(batch_size))
            else:
                pred_indices = [self._node2index[p] for p in v.predecessors]
                x[:, index] = self.layer[node](x[:, pred_indices]).squeeze(-1)
        return x

In [147]:
model_formula = "X <- A + B + C; B <- A + C; A ~ uniform; C ~ norm"
model = sem2(model_formula)
var = model._node2info
[var, 
 var["A"].dist.rvs(10), var["C"].dist.rvs(10), 
 var["A"].successors, var["B"].predecessors, model._topological_order]

[{'X': X<-A+B+C, 'B': B<-A+C, 'A': A~uniform, 'C': C~norm},
 array([0.9967725 , 0.86206064, 0.67425023, 0.70982414, 0.49056005,
        0.266223  , 0.12288002, 0.46959319, 0.09383777, 0.45506514]),
 array([ 0.36360987, -0.91211033, -1.35661684,  1.09964212,  0.53117628,
         0.5401992 ,  1.21273898, -0.4894036 ,  0.65908402,  1.9983124 ]),
 ['X', 'B'],
 ['A', 'C'],
 ['A', 'C', 'B', 'X']]

In [148]:
model(8)

tensor([[ 0.3378,  0.3512, -0.9461,  0.7267],
        [ 0.5424, -0.3710, -0.4690,  0.5964],
        [ 0.9968,  2.1794, -2.2621,  1.6735],
        [ 0.8402,  0.4701, -1.0739,  1.0292],
        [ 0.1482,  0.8313, -1.2584,  0.7852],
        [ 0.3236, -0.1656, -0.5900,  0.5493],
        [ 0.2407,  1.6655, -1.8397,  1.1081],
        [ 0.2891,  0.4707, -1.0237,  0.7405]], grad_fn=<CopySlices>)

In [200]:
# Linear factor model
model_formula = "x1 <- ind60 + ex1; x2 <- ind60 + ex2; x3 <- ind60 + ex3; ind60 ~ norm; ex1 ~ norm; ex2 ~ norm; ex3 ~ norm"
model = sem2(model_formula)
observed_var = ["x1", "x2", "x3"]
x = dataset[:, [var_names.index(var) for var in observed_var]]
observed_index = [list(model.static_order()).index(var) for var in observed_var]

In [201]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model.train()
for i in range(1000):
    optimizer.zero_grad()
    sample1 = model(x.size(0))[:, observed_index]
    sample2 = model(x.size(0))[:, observed_index]
    loss, s1, s2 = energy_loss_two_sample(x, sample1, sample2)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 100 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')

Iter 1 | loss: 7.0525, s1: 7.5722, s2: 1.0392
Iter 100 | loss: 4.7067, s1: 6.5842, s2: 3.7550
Iter 200 | loss: 2.6051, s1: 5.4864, s2: 5.7625
Iter 300 | loss: 2.7816, s1: 5.5154, s2: 5.4674
Iter 400 | loss: 1.8768, s1: 4.2777, s2: 4.8019
Iter 500 | loss: 1.5698, s1: 3.5780, s2: 4.0163
Iter 600 | loss: 1.5308, s1: 3.2732, s2: 3.4848
Iter 700 | loss: 1.3809, s1: 2.8689, s2: 2.9760
Iter 800 | loss: 1.2844, s1: 2.7487, s2: 2.9285
Iter 900 | loss: 1.2121, s1: 2.8551, s2: 3.2859
Iter 1000 | loss: 1.4553, s1: 2.7260, s2: 2.5414


In [205]:
beta = [model.layer[node].weight[0][0].squeeze().detach().numpy().tolist() for node in observed_var]
[beta[i] / beta[0] for i in range(3)]

[1.0, 2.0782818544013013, 1.8808241202779672]

In [248]:
# Linear factor model (same as lavaan package example)
formula_epsx = "ex1 ~ norm; ex2 ~ norm; ex3 ~ norm;" 
formula_epsy = "ey1 ~ norm; ey2 ~ norm; ey3 ~ norm; ey4 ~ norm; ey5 ~ norm; ey6 ~ norm; ey7 ~ norm; ey8 ~ norm;"
formula_epsdem = "edem60 ~ norm; edem65 ~ norm;"
formula_epscov = "e15 ~ norm; e24 ~ norm; e26 ~ norm; e37 ~ norm; e48 ~ norm; e68 ~ norm;"
formula_end = "ind60 ~ norm; dem60 <- ind60 + edem60; dem65 <- ind60 + dem60 + edem65;"
formula_x = "x1 <- ind60 + ex1; x2 <- ind60 + ex2; x3 <- ind60 + ex3;"
formula_y1 = "y1 <- dem60 + ey1 + e15; y2 <- dem60 + ey2 + e24 + e26; y3 <- dem60 + ey3 + e37; y4 <- dem60 + ey4 + e24 + e48;"
formula_y2 = "y5 <- dem65 + ey5 + e15; y6 <- dem65 + ey6 + e26; y7 <- dem65 + ey7 + e37; y8 <- dem65 + ey8 + e48 + e68"
formula = formula_epsx + formula_epsy + formula_epsdem + formula_epscov + formula_end + formula_x + formula_y1 + formula_y2

In [261]:
# Linear factor model
formula_epsx = "ex1 ~ norm; ex2 ~ norm; ex3 ~ norm;" 
formula_epsy = "ey1 ~ norm; ey2 ~ norm; ey3 ~ norm; ey4 ~ norm; ey5 ~ norm; ey6 ~ norm; ey7 ~ norm; ey8 ~ norm;"
formula_epsdem = "edem60 ~ norm; edem65 ~ norm;"
formula_end = "ind60 ~ norm; dem60 <- ind60 + edem60; dem65 <- ind60 + dem60 + edem65;"
formula_x = "x1 <- ind60 + ex1; x2 <- ind60 + ex2; x3 <- ind60 + ex3;"
formula_y1 = "y1 <- dem60 + ey1; y2 <- dem60 + ey2; y3 <- dem60 + ey3; y4 <- dem60 + ey4;"
formula_y2 = "y5 <- dem65 + ey5; y6 <- dem65 + ey6; y7 <- dem65 + ey7; y8 <- dem65 + ey8"
formula = formula_epsx + formula_epsy + formula_epsdem + formula_end + formula_x + formula_y1 + formula_y2

In [262]:
observed_var = var_names
x = dataset[:, [var_names.index(var) for var in observed_var]]
observed_index = [list(model.static_order()).index(var) for var in observed_var]

In [317]:
sample = model(x.size(0) * 10)[:, observed_index]
loss = energy_loss(x, sample)
loss

tensor([16.2868, 17.5314,  2.4893], grad_fn=<CatBackward0>)

In [323]:
model = sem2(formula)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model.train()
for i in range(20000):
    optimizer.zero_grad()
#    sample1 = model(x.size(0))[:, observed_index]
#    sample2 = model(x.size(0))[:, observed_index]
#    loss, s1, s2 = energy_loss_two_sample(x, sample1, sample2)
    sample = model(x.size(0) * 100)[:, observed_index]
    loss, s1, s2 = energy_loss(x, sample)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 500 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')
        ind60_scale = model.layer["x1"].weight[0][0]
        dem60_scale = model.layer["y1"].weight[0][0]
        dem65_scale = model.layer["y5"].weight[0][0]
        print([model.layer["dem60"].weight[0][0] / ind60_scale * dem60_scale,
               model.layer["dem65"].weight[0][0] / ind60_scale * dem65_scale,
               model.layer["dem65"].weight[0][1] / dem60_scale * dem65_scale])

Iter 1 | loss: 16.8636, s1: 17.9362, s2: 2.1452
[tensor(0.0559, grad_fn=<MulBackward0>), tensor(-0.7689, grad_fn=<MulBackward0>), tensor(-0.0332, grad_fn=<MulBackward0>)]
Iter 500 | loss: 6.4126, s1: 13.0525, s2: 13.2799
[tensor(1.5865, grad_fn=<MulBackward0>), tensor(1.3757, grad_fn=<MulBackward0>), tensor(0.5761, grad_fn=<MulBackward0>)]
Iter 1000 | loss: 6.2052, s1: 12.5538, s2: 12.6971
[tensor(3.1555, grad_fn=<MulBackward0>), tensor(1.8701, grad_fn=<MulBackward0>), tensor(0.6081, grad_fn=<MulBackward0>)]
Iter 1500 | loss: 6.1811, s1: 12.5701, s2: 12.7779
[tensor(3.0933, grad_fn=<MulBackward0>), tensor(1.6600, grad_fn=<MulBackward0>), tensor(0.6231, grad_fn=<MulBackward0>)]
Iter 2000 | loss: 6.1712, s1: 12.5225, s2: 12.7025
[tensor(3.0627, grad_fn=<MulBackward0>), tensor(1.4827, grad_fn=<MulBackward0>), tensor(0.6409, grad_fn=<MulBackward0>)]
Iter 2500 | loss: 6.1939, s1: 12.6048, s2: 12.8218
[tensor(2.4122, grad_fn=<MulBackward0>), tensor(1.3228, grad_fn=<MulBackward0>), tensor(0.6

In [308]:
# change loss function to OLS
sample_cov = torch.cov(torch.t(x))
model = sem2(formula)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model.train()
for i in range(20000):
    optimizer.zero_grad()
    model_sample = model(x.size(0) * 10)[:, observed_index]
    model_cov = torch.cov(torch.t(model_sample))
    loss = torch.norm(sample_cov - model_cov)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 500 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}')
        ind60_scale = model.layer["x1"].weight[0][0]
        dem60_scale = model.layer["y1"].weight[0][0]
        dem65_scale = model.layer["y5"].weight[0][0]
        print([model.layer["dem60"].weight[0][0] / ind60_scale * dem60_scale,
               model.layer["dem65"].weight[0][0] / ind60_scale * dem65_scale,
               model.layer["dem65"].weight[0][1] / dem60_scale * dem65_scale])

Iter 1 | loss: 58.9334
[tensor(0.2185, grad_fn=<MulBackward0>), tensor(-0.5193, grad_fn=<MulBackward0>), tensor(0.2891, grad_fn=<MulBackward0>)]
Iter 500 | loss: 10.5364
[tensor(3.5093, grad_fn=<MulBackward0>), tensor(1.9779, grad_fn=<MulBackward0>), tensor(0.5918, grad_fn=<MulBackward0>)]
Iter 1000 | loss: 6.8951
[tensor(3.2442, grad_fn=<MulBackward0>), tensor(1.6923, grad_fn=<MulBackward0>), tensor(0.6184, grad_fn=<MulBackward0>)]
Iter 1500 | loss: 6.3743
[tensor(3.0811, grad_fn=<MulBackward0>), tensor(1.5131, grad_fn=<MulBackward0>), tensor(0.6474, grad_fn=<MulBackward0>)]
Iter 2000 | loss: 6.2319
[tensor(2.7867, grad_fn=<MulBackward0>), tensor(1.3249, grad_fn=<MulBackward0>), tensor(0.6879, grad_fn=<MulBackward0>)]
Iter 2500 | loss: 6.0422
[tensor(2.4561, grad_fn=<MulBackward0>), tensor(1.1074, grad_fn=<MulBackward0>), tensor(0.7234, grad_fn=<MulBackward0>)]
Iter 3000 | loss: 8.3183
[tensor(2.0843, grad_fn=<MulBackward0>), tensor(0.9289, grad_fn=<MulBackward0>), tensor(0.7898, grad

In [331]:
# change loss function to negative log likelihood
sample_cov = torch.cov(torch.t(x))
model = sem2(formula)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model.train()
for i in range(20000):
    optimizer.zero_grad()
    model_sample = model(x.size(0) * 10)[:, observed_index]
    model_cov = torch.cov(torch.t(model_sample))
    loss = torch.log(torch.det(model_cov)) + torch.trace(sample_cov @ torch.inverse(model_cov))
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 500 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}')
        ind60_scale = model.layer["x1"].weight[0][0]
        dem60_scale = model.layer["y1"].weight[0][0]
        dem65_scale = model.layer["y5"].weight[0][0]
        print([model.layer["dem60"].weight[0][0] / ind60_scale * dem60_scale,
               model.layer["dem65"].weight[0][0] / ind60_scale * dem65_scale,
               model.layer["dem65"].weight[0][1] / dem60_scale * dem65_scale])

Iter 1 | loss: 3474.9500
[tensor(-1.2587, grad_fn=<MulBackward0>), tensor(0.1480, grad_fn=<MulBackward0>), tensor(-0.1256, grad_fn=<MulBackward0>)]
Iter 500 | loss: 64.1775
[tensor(4.1614, grad_fn=<MulBackward0>), tensor(1.6793, grad_fn=<MulBackward0>), tensor(0.4999, grad_fn=<MulBackward0>)]
Iter 1000 | loss: 42.1812
[tensor(2.6118, grad_fn=<MulBackward0>), tensor(1.1200, grad_fn=<MulBackward0>), tensor(0.5635, grad_fn=<MulBackward0>)]
Iter 1500 | loss: 35.0308
[tensor(1.9371, grad_fn=<MulBackward0>), tensor(0.8591, grad_fn=<MulBackward0>), tensor(0.5875, grad_fn=<MulBackward0>)]
Iter 2000 | loss: 29.1986
[tensor(1.6624, grad_fn=<MulBackward0>), tensor(0.7582, grad_fn=<MulBackward0>), tensor(0.6336, grad_fn=<MulBackward0>)]
Iter 2500 | loss: 27.1512
[tensor(1.6309, grad_fn=<MulBackward0>), tensor(0.7304, grad_fn=<MulBackward0>), tensor(0.6353, grad_fn=<MulBackward0>)]
Iter 3000 | loss: 25.9855
[tensor(1.5747, grad_fn=<MulBackward0>), tensor(0.7200, grad_fn=<MulBackward0>), tensor(0.64